# 01 · Data profiling (what is messy in this dataset?)

Looks at every table before any modelling: shapes, missing values, date formats, units, duplicates and odd category spellings.

**Needs:** the competition CSVs only. **Writes:** nothing (read-only exploration). **Time:** about 5 minutes.

Things this exploration found (and the later notebooks fix): lab units are mixed (g/L vs g/dL, µmol/L vs mg/dL, mmol/L vs mg/dL), temperature is a mix of °F and °C,
some lab values are decimal slips (sodium 1450), about 27% of ICD codes have lost their decimal point, drug names appear in several capitalisations,
`MEDICARE` vs `Medicare`, `?` as a race value, and exact duplicate rows in several tables.

**Setup.** Install the libraries once (uncomment), then set the two paths in the next cell.

In [ ]:
# %pip install -q lightgbm xgboost scikit-learn pandas pyarrow scipy

In [ ]:
import os, sys, json, time, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# ---- EDIT THESE TWO PATHS --------------------------------------------------
DATA = os.environ.get('KAGGLE_DATA', 'data')   # folder holding the unzipped competition CSVs (train.csv, test.csv, patients.csv, ...)
WORK = os.environ.get('KAGGLE_WORK', 'work')   # folder where features, predictions and submissions are saved (shared by all notebooks)
# -----------------------------------------------------------------------------
Path(WORK).mkdir(parents=True, exist_ok=True)
print('DATA ->', Path(DATA).resolve()); print('WORK ->', Path(WORK).resolve())

### 1.1 Train/test, encounters, patients, hospitals

In [ ]:
import pandas as pd, sys
D=DATA
pd.set_option('display.width',200); pd.set_option('display.max_columns',30)
tr=pd.read_csv(f'{D}/train.csv'); te=pd.read_csv(f'{D}/test.csv')
ie=pd.read_csv(f'{D}/index_encounters.csv')
print('train',tr.shape,tr.readmit_30d.mean(),'test',te.shape, 'dups',tr.encounter_id.duplicated().sum(), te.encounter_id.duplicated().sum(), 'overlap',len(set(tr.encounter_id)&set(te.encounter_id)))
print('ie',ie.shape,'dup enc',ie.encounter_id.duplicated().sum(),'dup pat',ie.patient_id.duplicated().sum())
print('in ie: train',tr.encounter_id.isin(ie.encounter_id).mean(),'test',te.encounter_id.isin(ie.encounter_id).mean())
print(ie.isna().mean())
for c in ['admission_type','admission_source','service_line','discharge_disposition','attending_specialty']:
    print(ie[c].value_counts(dropna=False).head(25))
print(ie.admit_dt.sample(15,random_state=1).tolist()); print(ie.discharge_dt.sample(15,random_state=1).tolist())
print(ie.admit_weight_kg.describe())
p=pd.read_csv(f'{D}/patients.csv'); print(p.shape, p.patient_id.duplicated().sum()); print(p.isna().mean())
for c in ['sex','race_ethnicity','language','marital_status','insurance_type']:
    print(p[c].value_counts(dropna=False).head(20))
print(p.birth_year.describe()); print(p.deprivation_index.describe()); print(p.zip3.nunique())
h=pd.read_csv(f'{D}/hospitals.csv'); print(h.describe(include='all'))

### 1.2 Diagnoses, procedures, medications, discharge planning, transfers, prior visits

In [ ]:
import pandas as pd, sys
D=DATA
pd.set_option('display.width',200); pd.set_option('display.max_columns',30); pd.set_option('display.max_rows',200)
ie=pd.read_csv(f'{D}/index_encounters.csv',parse_dates=['admit_dt','discharge_dt'])
los=(ie.discharge_dt-ie.admit_dt).dt.total_seconds()/86400
print('LOS',los.describe(), (los<0).sum())
p=pd.read_csv(f'{D}/patients.csv'); print((p.birth_year<=1900).sum(), p.birth_year.value_counts().head(3))
dx=pd.read_csv(f'{D}/diagnoses.csv'); print('dx',dx.shape,'enc',dx.encounter_id.nunique(),'dups',dx.duplicated().sum(),'dup enc/seq',dx.duplicated(['encounter_id','seq']).sum())
print(dx.isna().mean()); print(dx.present_on_admission.value_counts(dropna=False)); print(dx.is_primary.value_counts(dropna=False))
lk=pd.read_csv(f'{D}/icd10_lookup.csv'); print('dx codes not in lookup:',(~dx.icd10_code.isin(lk.icd10_code)).mean()); print(dx.icd10_code[~dx.icd10_code.isin(lk.icd10_code)].value_counts().head(15))
print(dx.groupby('encounter_id').is_primary.sum().value_counts().head())
pr=pd.read_csv(f'{D}/procedures.csv'); print('proc',pr.shape,pr.encounter_id.nunique()); print(pr.code_type.value_counts(dropna=False)); print(pr.code.value_counts().head(10)); print(pr.proc_dt.sample(8,random_state=0).tolist())
md=pd.read_csv(f'{D}/medications.csv'); print('med',md.shape,md.encounter_id.nunique(),'dups',md.duplicated().sum()); print(md.isna().mean()); print(md.drug_class.value_counts(dropna=False)); print(md.route.value_counts(dropna=False)); print(md.continued_at_discharge.value_counts(dropna=False)); print(md.drug_name.nunique(), md.drug_name.value_counts().head(8))
dp=pd.read_csv(f'{D}/discharge_planning.csv'); print('dp',dp.shape,dp.encounter_id.nunique()); print(dp.isna().mean())
for c in ['home_health_referral','med_reconciliation_done','education_done']: print(dp[c].value_counts(dropna=False))
print(dp.followup_scheduled_days.describe())
ut=pd.read_csv(f'{D}/unit_transfers.csv'); print('ut',ut.shape,ut.encounter_id.nunique()); print(ut.unit.value_counts(dropna=False)); print(ut.isna().mean()); print(ut.in_dt.sample(5,random_state=0).tolist())
pu=pd.read_csv(f'{D}/prior_utilization.csv'); print('pu',pu.shape,pu.patient_id.nunique()); print(pu.visit_type.value_counts(dropna=False)); print(pu.visit_dt.sample(8,random_state=0).tolist()); print(pu.isna().mean()); print('dups',pu.duplicated().sum())

### 1.3 Lab results and vitals (units and outliers)

In [ ]:
import pandas as pd, sys
D=DATA
pd.set_option('display.width',220); pd.set_option('display.max_columns',30); pd.set_option('display.max_rows',300)
lb=pd.read_csv(f'{D}/lab_results.csv'); print(lb.shape, lb.encounter_id.nunique(), 'dups',lb.duplicated().sum()); print(lb.isna().mean())
print(lb.groupby(['lab_code','lab_name'],dropna=False).size())
print(lb.groupby(['lab_name','unit'],dropna=False).value.agg(['count','mean','median','min','max']))
print(lb.abnormal_flag.value_counts(dropna=False))
print(lb.collected_dt.sample(12,random_state=2).tolist())
vt=pd.read_csv(f'{D}/vitals.csv'); print(vt.shape, vt.encounter_id.nunique(),'dups',vt.duplicated().sum()); print(vt.isna().mean())
print(vt.groupby('vital_type',dropna=False).value.describe(percentiles=[.01,.5,.99]))
print(vt.charttime.sample(12,random_state=2).tolist())
print(vt.vital_type.value_counts(dropna=False))

### 1.4 Timing checks (are labs/vitals/prior visits all inside the allowed windows?)
Result: yes. Nothing is dated after discharge and every prior visit is before admission.

In [ ]:
import pandas as pd, numpy as np, sys
D=DATA
ie=pd.read_csv(f'{D}/index_encounters.csv',parse_dates=['admit_dt','discharge_dt'])[['encounter_id','patient_id','admit_dt','discharge_dt']]
lb=pd.read_csv(f'{D}/lab_results.csv',parse_dates=['collected_dt']).merge(ie,on='encounter_id')
dh=(lb.collected_dt-lb.admit_dt).dt.total_seconds()/3600; dd=(lb.discharge_dt-lb.collected_dt).dt.total_seconds()/3600
print('labs before admit',(dh<0).mean(),'after discharge',(dd<0).mean()); print(dh.describe()); print(dd.describe())
vt=pd.read_csv(f'{D}/vitals.csv'); vt['t']=pd.to_datetime(vt.charttime,format='%m/%d/%Y %H:%M'); vt=vt.merge(ie,on='encounter_id')
dh=(vt.t-vt.admit_dt).dt.total_seconds()/3600; dd=(vt.discharge_dt-vt.t).dt.total_seconds()/3600
print('vit before admit',(dh<0).mean(),'after dc',(dd<0).mean()); print(dh.describe()); print(dd.describe())
print(vt.groupby('encounter_id').size().describe())
pu=pd.read_csv(f'{D}/prior_utilization.csv',parse_dates=['visit_dt']).merge(ie,on='patient_id')
print('prior after admit',(pu.visit_dt>=pu.admit_dt).mean()); print(((pu.admit_dt-pu.visit_dt).dt.days).describe())
ut=pd.read_csv(f'{D}/unit_transfers.csv',parse_dates=['in_dt','out_dt']).merge(ie,on='encounter_id')
print('ut in<admit',(ut.in_dt<ut.admit_dt-pd.Timedelta('1min')).mean(),'out>dc',(ut.out_dt>ut.discharge_dt+pd.Timedelta('1min')).mean(), 'neg dur',(ut.out_dt<ut.in_dt).mean())
pr=pd.read_csv(f'{D}/procedures.csv'); pr['d']=pd.to_datetime(pr.proc_dt,format='%d-%b-%Y').dt.normalize(); pr=pr.merge(ie,on='encounter_id')
print('proc outside stay',((pr.d<pr.admit_dt.dt.normalize())|(pr.d>pr.discharge_dt.dt.normalize())).mean())
print(ie.admit_dt.dt.year.value_counts().sort_index())